# Azure Content Understanding - API Testing Guide (Python)

This notebook replicates all the API tests from the `.http` test files using **Python** with `httpx` and `azure-identity`.

**Documentation:**
- [Overview](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/overview)
- [REST API Reference](https://learn.microsoft.com/en-us/rest/api/contentunderstanding/operation-groups)

## Table of Contents
1. **Setup** – Install packages, load credentials, create HTTP client
2. **Quick Start** – Check defaults, layout extraction, invoice analysis
3. **Section 1** – Content Extraction (no LLM required)
4. **Section 2** – Domain-Specific Analyzers (invoice, document)
5. **Section 3** – RAG Analyzers (documentSearch, imageSearch, etc.)
6. **Section 4** – Custom Analyzers with Field Extraction
7. **Section 5** – Custom Video Analysis with Field Extraction
8. **Section 6** – Analyzer Management (List, Copy, Delete)
9. **Bonus** – Custom Analyzer with allowReplace

## Setup

Authenticate via **Microsoft Entra ID** using `DefaultAzureCredential`.

**Before running this notebook:**
1. Run `az login` in your terminal (authenticates Azure CLI)
2. Ensure you have the **Cognitive Services User** role on the CU resource (ask your infra team to assign it)

Set the following environment variables (or use a `.env` file):
```
ENDPOINT_URL=https://your-resource.cognitiveservices.azure.com
API_VERSION=2025-11-01
AOAI_CONNECTION=your-aoai-connection-name
GPT41_DEPLOYMENT=gpt-41
GPT41_MINI_DEPLOYMENT=gpt-41-mini
EMBEDDING_ADA_DEPLOYMENT=text-embedding-ada-002
EMBEDDING_3LARGE_DEPLOYMENT=text-embedding-3-large
```

> **Note:** No API key is needed — authentication uses your Azure CLI login token via `DefaultAzureCredential`.

**Prerequisites:** Python 3.10+, Azure CLI (`az login`), packages from `requirements.txt`.

In [ ]:
# Install required packages (run once)
%pip install azure-identity httpx --quiet

In [ ]:
import os
import json
import time
import httpx
from azure.identity import DefaultAzureCredential

# Load configuration from environment variables
ENDPOINT = os.environ.get("ENDPOINT_URL", "").rstrip("/")
if not ENDPOINT:
    raise RuntimeError("ENDPOINT_URL environment variable must be set")

API_VERSION = os.environ.get("API_VERSION", "2025-11-01")
AOAI_CONNECTION = os.environ.get("AOAI_CONNECTION", "")
if not AOAI_CONNECTION:
    raise RuntimeError("AOAI_CONNECTION environment variable must be set")

GPT41_DEPLOYMENT = os.environ.get("GPT41_DEPLOYMENT", "gpt-41")
GPT41_MINI_DEPLOYMENT = os.environ.get("GPT41_MINI_DEPLOYMENT", "gpt-41-mini")
EMBEDDING_ADA_DEPLOYMENT = os.environ.get("EMBEDDING_ADA_DEPLOYMENT", "text-embedding-ada-002")
EMBEDDING_3LARGE_DEPLOYMENT = os.environ.get("EMBEDDING_3LARGE_DEPLOYMENT", "text-embedding-3-large")

# Authenticate via Entra ID (DefaultAzureCredential chains: AzureCLI -> ManagedIdentity -> etc.)
credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default")

client = httpx.Client(
    headers={"Authorization": f"Bearer {token.token}"},
    timeout=60.0,
)

print(f"Endpoint: {ENDPOINT}")
print(f"API Version: {API_VERSION}")
print(f"AOAI Connection: {AOAI_CONNECTION}")
print(f"Token expires: {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime(token.expires_on))} (re-run this cell to refresh)")
print("Credentials loaded successfully (Entra ID).")

In [ ]:
def pretty(data):
    """Pretty-print a dict/list as indented JSON."""
    print(json.dumps(data, indent=2, ensure_ascii=False))


def analyze_and_poll(analyzer_id: str, payload: dict, poll_interval: int = 5, max_wait: int = 300) -> dict:
    """Submit an analyze request and poll until completion."""
    url = f"{ENDPOINT}/contentunderstanding/analyzers/{analyzer_id}:analyze?api-version={API_VERSION}"
    resp = client.post(url, json=payload)
    resp.raise_for_status()

    operation = resp.json()
    operation_id = operation["id"]
    status = operation.get("status", "unknown")
    print(f"Operation started: {operation_id}")
    print(f"Status: {status}")

    result_url = f"{ENDPOINT}/contentunderstanding/analyzerResults/{operation_id}?api-version={API_VERSION}"
    elapsed = 0
    result = None

    while elapsed < max_wait:
        time.sleep(poll_interval)
        elapsed += poll_interval

        result_resp = client.get(result_url)
        result_resp.raise_for_status()
        result = result_resp.json()
        status = result.get("status", "unknown")
        print(f"  [{elapsed}s] Status: {status}")

        if status.lower() in ("succeeded", "failed"):
            return result

    print(f"Timed out after {max_wait}s. Returning last response.")
    return result

---
## Quick Start: Get Started in 3 Steps

1. Check model deployments (GPT-4.1, GPT-4.1-mini, text-embedding-ada-002)
2. Try content extraction (no LLM required)
3. Try a domain-specific analyzer (invoice)

### Step 1: Check current model deployment settings

In [ ]:
resp = client.get(f"{ENDPOINT}/contentunderstanding/defaults?api-version={API_VERSION}")
resp.raise_for_status()
pretty(resp.json())

### Step 1b: Configure model deployments (if needed)

Only required if using BYOC (Bring Your Own Compute) with custom Azure OpenAI deployments.  
Update the deployment names below to match your setup, then run the cell.

In [ ]:
patch_body = {
    "modelDeployments": {
        "gpt-4.1": f"{AOAI_CONNECTION}/{GPT41_DEPLOYMENT}",
        "text-embedding-ada-002": f"{AOAI_CONNECTION}/{EMBEDDING_ADA_DEPLOYMENT}",
        "text-embedding-3-large": f"{AOAI_CONNECTION}/{EMBEDDING_3LARGE_DEPLOYMENT}",
        "gpt-4.1-mini": f"{AOAI_CONNECTION}/{GPT41_MINI_DEPLOYMENT}",
        "prebuilt-analyzer-completion": f"{AOAI_CONNECTION}/{GPT41_DEPLOYMENT}",
        "prebuilt-analyzer-embedding": f"{AOAI_CONNECTION}/{EMBEDDING_3LARGE_DEPLOYMENT}",
        "prebuilt-analyzer-completion-mini": f"{AOAI_CONNECTION}/{GPT41_MINI_DEPLOYMENT}",
    }
}

resp = client.patch(
    f"{ENDPOINT}/contentunderstanding/defaults?api-version={API_VERSION}",
    json=patch_body,
)
resp.raise_for_status()
pretty(resp.json())

### Step 2: Content extraction with prebuilt-layout

This analyzer does **NOT** require LLM models – works immediately.  
Extracts text, tables, figures, sections, hyperlinks, and annotations.

In [ ]:
result = analyze_and_poll("prebuilt-layout", {
    "inputs": [{"url": "https://github.com/Azure-Samples/cognitive-services-sample-data-files/raw/master/ComputerVision/Images/printed_text.jpg"}]
})
pretty(result)

### Step 3: Domain-specific analyzer – prebuilt-invoice

Extracts vendor, items, totals, dates from invoices.  
Uses the LLM in default/models for field extraction.

In [ ]:
result = analyze_and_poll("prebuilt-invoice", {
    "inputs": [{"url": "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf"}]
})
pretty(result)

---
## Section 1: Content Extraction (No LLM Required)

These analyzers do **NOT** require LLM or embedding models. They work immediately after resource creation.

- **prebuilt-read** – Basic OCR (words, paragraphs, formulas, barcodes)
- **prebuilt-layout** – Advanced OCR with layout (tables, figures, sections)

### 1.1 prebuilt-read: Basic OCR

Extracts words, paragraphs, formulas, barcodes without layout analysis.

In [ ]:
result = analyze_and_poll("prebuilt-read", {
    "inputs": [{"url": "https://github.com/Azure-Samples/cognitive-services-sample-data-files/raw/master/ComputerVision/Images/handwritten_text.jpg"}]
})
pretty(result)

### 1.2 prebuilt-layout: OCR with layout analysis

Extracts text, tables, figures, sections, hyperlinks, annotations.

In [ ]:
result = analyze_and_poll("prebuilt-layout", {
    "inputs": [{"url": "https://github.com/Azure-Samples/cognitive-services-sample-data-files/raw/master/ComputerVision/Images/printed_text.jpg"}]
})
pretty(result)

---
## Section 2: Domain-Specific Analyzers

Preconfigured analyzers for common document types. Powered by rich knowledge bases of real-world examples.

Available analyzers:
- **Financial:** prebuilt-invoice, prebuilt-receipt, prebuilt-creditCard
- **Identity:** prebuilt-idDocument, prebuilt-healthInsuranceCard.us
- **Tax (US):** prebuilt-tax.us.1040, prebuilt-tax.us.w2, prebuilt-tax.us.1099*
- **Mortgage (US):** prebuilt-mortgage.us.1003, prebuilt-mortgage.us.1004
- **Other:** prebuilt-utilityBill, prebuilt-payStub.us, prebuilt-contract

### 2.1 prebuilt-invoice

In [ ]:
result = analyze_and_poll("prebuilt-invoice", {
    "inputs": [{"url": "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf"}]
})
pretty(result)

### 2.2 prebuilt-document

General-purpose document analysis with tables, key-value pairs, entities.  
Works with any document type – no domain-specific training needed.

In [ ]:
result = analyze_and_poll("prebuilt-document", {
    "inputs": [{"url": "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/mixed_financial_docs.pdf"}]
})
pretty(result)

---
## Section 3: RAG Analyzers (Requires LLM + Embeddings)

Optimized for RAG scenarios with semantic analysis and markdown extraction.

- **prebuilt-documentSearch** – Document ingestion with summaries
- **prebuilt-imageSearch** – Image analysis with descriptions
- **prebuilt-audioSearch** – Audio transcription with summaries
- **prebuilt-videoSearch** – Video segmentation with transcripts

> **Tip:** Use the `range` parameter to limit analysis to specific pages (e.g., `"1-3"`, `"1,5,7"`).

### 3.1 prebuilt-documentSearch

Extracts content, tables, figures with descriptions, chart.js/mermaid.js output.  
Generates document summaries, captures annotations.

In [ ]:
result = analyze_and_poll("prebuilt-documentSearch", {
    "inputs": [{
        "url": "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data/refs/heads/main/sustainable-ai-pdf/Accelerating-Sustainability-with-AI-2025.pdf",
        "range": "1-3"
    }]
})
pretty(result)

### 3.1b Get RAG analyzer definition

In [ ]:
resp = client.get(f"{ENDPOINT}/contentunderstanding/analyzers/prebuilt-documentSearch?api-version={API_VERSION}")
resp.raise_for_status()
pretty(resp.json())

---
## Section 4: Custom Analyzers with Field Extraction

**Field Extraction** is the core value of custom analyzers. Define a field schema to extract specific structured data.

Field extraction methods:
- **extract** – Extract existing data from the content (e.g., policy number)
- **classify** – Classify content into predefined categories (e.g., loss type)
- **generate** – Generate new insights from the content (e.g., summaries)

### 4.1 Create custom invoice analyzer with field extraction

In [ ]:
custom_invoice_schema = {
    "baseAnalyzerId": "prebuilt-document",
    "analyzerId": "custom_invoice",
    "models": {"completion": "gpt-4.1", "embedding": "text-embedding-ada-002"},
    "fieldSchema": {
        "fields": {
            "InvoiceNumber": {"type": "string", "method": "extract", "description": "The invoice number (e.g., INV-100)"},
            "InvoiceDate": {"type": "string", "method": "extract", "description": "The invoice date"},
            "CustomerName": {"type": "string", "method": "extract", "description": "The customer name or company name"},
            "TotalAmount": {"type": "number", "method": "extract", "description": "The total amount due on the invoice"},
            "Subtotal": {"type": "number", "method": "extract", "description": "The subtotal before taxes"},
            "SalesTax": {"type": "number", "method": "extract", "description": "The sales tax amount"},
            "LineItems": {
                "type": "array",
                "method": "extract",
                "description": "Individual line items from the invoice",
                "items": {
                    "type": "object",
                    "properties": {
                        "Date": {"type": "string", "description": "Date of the service or item"},
                        "Description": {"type": "string", "description": "Description of the service or item"},
                        "Quantity": {"type": "number", "description": "Quantity or hours"},
                        "Price": {"type": "number", "description": "Unit price"},
                        "Amount": {"type": "number", "description": "Line item total amount"}
                    }
                }
            }
        },
        "definitions": {}
    },
    "omitContent": True
}

resp = client.put(
    f"{ENDPOINT}/contentunderstanding/analyzers/custom_invoice?api-version={API_VERSION}",
    json=custom_invoice_schema,
)
print(f"Status: {resp.status_code}")
pretty(resp.json())

### Get custom invoice analyzer definition

In [ ]:
resp = client.get(f"{ENDPOINT}/contentunderstanding/analyzers/custom_invoice?api-version={API_VERSION}")
resp.raise_for_status()
pretty(resp.json())

### 4.2 Test the custom invoice analyzer

In [ ]:
result = analyze_and_poll("custom_invoice", {
    "inputs": [{"url": "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf"}]
})
pretty(result)

---
## Section 5: Custom Video Analysis with Field Extraction

Video analyzers demonstrate advanced field extraction with nested structures.  
Extract segments, scenes, timestamps, and generated insights from video content.

> **Note:** Video processing takes 1–2× the video duration. A 2-minute video may take 2–4 minutes to complete.

### 5.1 Create custom video analyzer (dynamic chaptering)

In [ ]:
video_schema = {
    "description": "Dynamic video chaptering with scene detection",
    "scenario": "videoShot",
    "baseAnalyzerId": "prebuilt-video",
    "models": {"completion": "gpt-4.1"},
    "config": {
        "returnDetails": True,
        "enableSegmentation": True,
        "locales": ["en-US"]
    },
    "fieldSchema": {
        "name": "Content Understanding - Dynamic Chaptering",
        "fields": {
            "Segments": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "SegmentId": {"type": "string"},
                        "SegmentType": {"type": "string", "method": "generate", "description": "The short title or a short summary of the story or chapter."},
                        "Scenes": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "Description": {"type": "string", "method": "generate", "description": "A five-word description of the scene."},
                                    "StartTimestamp": {"type": "string", "description": "the start timestamp of the scene"},
                                    "EndTimestamp": {"type": "string", "description": "the end timestamp of the scene"}
                                }
                            }
                        }
                    }
                }
            }
        }
    }
}

resp = client.put(
    f"{ENDPOINT}/contentunderstanding/analyzers/video_scenes_list?api-version={API_VERSION}",
    json=video_schema,
)
print(f"Status: {resp.status_code}")
pretty(resp.json())

### Get video analyzer definition

In [ ]:
resp = client.get(f"{ENDPOINT}/contentunderstanding/analyzers/video_chaptering?api-version={API_VERSION}")
resp.raise_for_status()
pretty(resp.json())

### 5.2 Test video analyzer

> Video analysis may take several minutes depending on video length.

In [ ]:
result = analyze_and_poll("video_chaptering", {
    "inputs": [{"url": "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/FlightSimulator.mp4"}]
}, poll_interval=15, max_wait=600)
pretty(result)

---
## Section 6: Analyzer Management (List, Copy, Delete)

### 6.1 List all available analyzers

In [ ]:
resp = client.get(f"{ENDPOINT}/contentunderstanding/analyzers?api-version={API_VERSION}")
resp.raise_for_status()
pretty(resp.json())

### 6.2 List only prebuilt analyzers

In [ ]:
resp = client.get(
    f"{ENDPOINT}/contentunderstanding/analyzers?api-version={API_VERSION}",
    params={"$filter": "startswith(analyzerId,'prebuilt-')"},
)
resp.raise_for_status()
pretty(resp.json())

### 6.3 Get specific analyzer definition (prebuilt-invoice)

In [ ]:
resp = client.get(f"{ENDPOINT}/contentunderstanding/analyzers/prebuilt-invoice?api-version={API_VERSION}")
resp.raise_for_status()
pretty(resp.json())

### 6.4 Copy analyzer within same resource

Useful for versioning or creating variations.

In [ ]:
copy_body = {"sourceAnalyzerId": "video_scenes_list"}

resp = client.post(
    f"{ENDPOINT}/contentunderstanding/analyzers/video_scenes_list-v2:copy?api-version={API_VERSION}",
    json=copy_body,
)
print(f"Status: {resp.status_code}")
pretty(resp.json())

### 6.5 Copy prebuilt to create stable version

Lock a prebuilt analyzer definition to prevent changes across API versions.

In [ ]:
copy_body = {"sourceAnalyzerId": "prebuilt-invoice"}

resp = client.post(
    f"{ENDPOINT}/contentunderstanding/analyzers/my-invoice:copy?api-version={API_VERSION}",
    json=copy_body,
)
print(f"Status: {resp.status_code}")
pretty(resp.json())

### 6.6 Delete custom analyzer

In [ ]:
resp = client.delete(f"{ENDPOINT}/contentunderstanding/analyzers/my-invoice?api-version={API_VERSION}")
print(f"Status: {resp.status_code}")
body = resp.text
if body.strip():
    pretty(resp.json())
else:
    print("Analyzer deleted successfully.")

---
## Bonus: Custom Analyzer with allowReplace

The `allowReplace=true` query parameter enables creating or replacing an existing analyzer with the same ID.  
This is useful for updating analyzer schemas without conflicts.

### Create custom claims analyzer (with allowReplace=true)

In [ ]:
claims_schema_v1 = {
    "description": "Custom analyzer for insurance claims processing",
    "tags": {
        "version": "1.0",
        "category": "claims"
    },
    "baseAnalyzerId": "prebuilt-document",
    "models": {"completion": "gpt-4.1", "embedding": "text-embedding-3-large"},
    "fieldSchema": {
        "name": "Insurance Claims Schema",
        "description": "Extract key information from insurance claims",
        "fields": {
            "ClaimNumber": {"type": "string", "method": "extract", "description": "The unique claim reference number"},
            "ClaimDate": {"type": "date", "method": "extract", "description": "The date the claim was filed"},
            "PolicyHolder": {"type": "string", "method": "extract", "description": "Name of the policy holder"},
            "PolicyNumber": {"type": "string", "method": "extract", "description": "The insurance policy number"},
            "LossType": {
                "type": "string", "method": "classify", "description": "Category of loss",
                "enum": ["property", "auto", "health", "liability", "other"]
            },
            "ClaimAmount": {"type": "number", "method": "extract", "description": "The amount claimed"},
            "ClaimSummary": {"type": "string", "method": "generate", "description": "AI-generated summary of the claim details"},
            "DocumentedExpenses": {
                "type": "array", "method": "extract",
                "description": "List of expenses documented in the claim",
                "items": {
                    "type": "object",
                    "properties": {
                        "Date": {"type": "date", "description": "Date of the expense"},
                        "Category": {"type": "string", "description": "Category of expense"},
                        "Amount": {"type": "number", "description": "Amount of the expense"},
                        "Description": {"type": "string", "description": "Details about the expense"}
                    }
                }
            }
        },
        "definitions": {}
    },
    "config": {
        "enableOcr": True,
        "enableLayout": True,
        "enableFormula": False,
        "returnDetails": True,
        "omitContent": False
    }
}

resp = client.put(
    f"{ENDPOINT}/contentunderstanding/analyzers/custom_claims_analyzer?api-version={API_VERSION}&allowReplace=true",
    json=claims_schema_v1,
)
print(f"Status: {resp.status_code}")
pretty(resp.json())

### Get custom claims analyzer definition

In [ ]:
resp = client.get(f"{ENDPOINT}/contentunderstanding/analyzers/custom_claims_analyzer?api-version={API_VERSION}")
resp.raise_for_status()
pretty(resp.json())

### Test the custom claims analyzer

In [ ]:
result = analyze_and_poll("custom_claims_analyzer", {
    "inputs": [{"url": "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf"}]
})
pretty(result)

### Update claims analyzer to v2 (with allowReplace=true)

Adds `ClaimStatus`, `AdjusterName`, `RiskAssessment` fields and `Verified` to expenses.

In [ ]:
claims_schema_v2 = {
    "description": "Updated custom analyzer for insurance claims processing - v2",
    "tags": {
        "version": "2.0",
        "category": "claims",
        "lastUpdated": "2025-01-29"
    },
    "baseAnalyzerId": "prebuilt-document",
    "models": {"completion": "gpt-4.1", "embedding": "text-embedding-3-large"},
    "fieldSchema": {
        "name": "Insurance Claims Schema v2",
        "description": "Enhanced schema for insurance claims with additional fields",
        "fields": {
            "ClaimNumber": {"type": "string", "method": "extract", "description": "The unique claim reference number"},
            "ClaimDate": {"type": "date", "method": "extract", "description": "The date the claim was filed"},
            "PolicyHolder": {"type": "string", "method": "extract", "description": "Name of the policy holder"},
            "PolicyNumber": {"type": "string", "method": "extract", "description": "The insurance policy number"},
            "LossType": {
                "type": "string", "method": "classify", "description": "Category of loss",
                "enum": ["property", "auto", "health", "liability", "other"]
            },
            "ClaimAmount": {"type": "number", "method": "extract", "description": "The amount claimed"},
            "ClaimStatus": {
                "type": "string", "method": "classify", "description": "Current status of the claim",
                "enum": ["pending", "approved", "denied", "settled"]
            },
            "AdjusterName": {"type": "string", "method": "extract", "description": "Name of the assigned claims adjuster"},
            "ClaimSummary": {"type": "string", "method": "generate", "description": "AI-generated summary of the claim details"},
            "RiskAssessment": {"type": "string", "method": "generate", "description": "AI-generated risk assessment based on claim details"},
            "DocumentedExpenses": {
                "type": "array", "method": "extract",
                "description": "List of expenses documented in the claim",
                "items": {
                    "type": "object",
                    "properties": {
                        "Date": {"type": "date", "description": "Date of the expense"},
                        "Category": {"type": "string", "description": "Category of expense"},
                        "Amount": {"type": "number", "description": "Amount of the expense"},
                        "Description": {"type": "string", "description": "Details about the expense"},
                        "Verified": {"type": "boolean", "description": "Whether the expense has been verified"}
                    }
                }
            }
        },
        "definitions": {}
    },
    "config": {
        "enableOcr": True,
        "enableLayout": True,
        "enableFormula": False,
        "returnDetails": True,
        "omitContent": False,
        "estimateFieldSourceAndConfidence": True
    }
}

resp = client.put(
    f"{ENDPOINT}/contentunderstanding/analyzers/custom_claims_analyzer?api-version={API_VERSION}&allowReplace=true",
    json=claims_schema_v2,
)
print(f"Status: {resp.status_code}")
pretty(resp.json())

### Create analyzer without allowReplace (strict creation)

Will return 201 if successful, or fail with a conflict error if the analyzer already exists.

In [ ]:
simple_schema = {
    "description": "Custom analyzer without allow replace",
    "baseAnalyzerId": "prebuilt-document",
    "models": {"completion": "gpt-4.1"},
    "fieldSchema": {
        "name": "Simple Schema",
        "fields": {
            "Title": {"type": "string", "method": "extract", "description": "Document title"}
        },
        "definitions": {}
    }
}

resp = client.put(
    f"{ENDPOINT}/contentunderstanding/analyzers/custom_new_analyzer?api-version={API_VERSION}",
    json=simple_schema,
)
print(f"Status: {resp.status_code}")
pretty(resp.json())